## EV Charging Coverage Analysis

This notebook provides a high-level analysis of registered electric vehicles and public charging infrastructure across Victoria. It uses the Gold postcode-level dataset to summarise the current statewide position, compare EV and charging coverage between regions, identify postcodes with high EV adoption, and highlight areas where charging availability may not align with demand. The analysis is intended to provide a simple and understandable starting point for transport planning and further infrastructure assessment.

In [14]:
%sql

--1. Current statewide position

SELECT
    SUM(registered_ev_count) AS registered_evs,
    SUM(charging_site_count) AS charging_sites,
    SUM(total_plug_count) AS charging_plugs,
    SUM(public_dc_site_count) AS public_dc_sites,
    SUM(public_dc_plug_count) AS public_dc_plugs,
    ROUND(
        SUM(registered_ev_count) /
        NULLIF(SUM(public_dc_plug_count), 0),
        1
    ) AS evs_per_public_dc_plug
FROM transport_planning.default.gold_ev_charging_infrastructure_by_postcode;

In [13]:
%sql

--2. EVs and charging infrastructure by region

WITH postcode_regions AS (
    SELECT
        EXPLODE_OUTER(ARRAY_DISTINCT(charging_regions)) AS region,
        registered_ev_count,
        charging_site_count,
        public_dc_site_count,
        total_plug_count,
        public_dc_plug_count
    FROM transport_planning.default.gold_ev_charging_infrastructure_by_postcode
)

SELECT
    COALESCE(region, 'Unassigned') AS region,
    SUM(registered_ev_count) AS registered_evs,
    SUM(charging_site_count) AS charging_sites,
    SUM(public_dc_site_count) AS public_dc_sites,
    SUM(total_plug_count) AS charging_plugs,
    SUM(public_dc_plug_count) AS public_dc_plugs,
    ROUND(
        SUM(registered_ev_count) /
        NULLIF(SUM(public_dc_plug_count), 0),
        1
    ) AS evs_per_public_dc_plug
FROM postcode_regions
GROUP BY COALESCE(region, 'Unassigned')
ORDER BY registered_evs DESC;

In [12]:
%sql

--3. Postcodes with the most EVs
    
SELECT
    postcode,
    charging_localities,
    charging_regions,
    registered_ev_count,
    ev_share_pct,
    charging_site_count,
    public_dc_site_count,
    public_dc_plug_count,
    evs_per_public_dc_plug,
    charging_coverage_status
FROM transport_planning.default.gold_ev_charging_infrastructure_by_postcode
ORDER BY registered_ev_count DESC
LIMIT 20;

In [10]:

--4. Postcodes with EVs but no public DC charging

SELECT
    postcode,
    charging_localities,
    charging_regions,
    registered_ev_count,
    ev_share_pct,
    benchmark_required_dc_plugs,
    public_dc_plug_shortfall,
    charging_coverage_status
FROM transport_planning.default.gold_ev_charging_infrastructure_by_postcode
WHERE registered_ev_count > 0
  AND has_public_dc_charging = FALSE
ORDER BY registered_ev_count DESC;

In [11]:
%sql

--5. Postcodes with the highest EV demand per public DC plug

SELECT
    postcode,
    charging_localities,
    charging_regions,
    registered_ev_count,
    public_dc_site_count,
    public_dc_plug_count,
    evs_per_public_dc_plug,
    benchmark_required_dc_plugs,
    public_dc_plug_shortfall,
    charging_coverage_status
FROM transport_planning.default.gold_ev_charging_infrastructure_by_postcode
WHERE public_dc_plug_count > 0
ORDER BY evs_per_public_dc_plug DESC
LIMIT 20;